In [1]:
import pandas as pd

from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

In [2]:
def drop_unnecessary(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=["native-country", "education", "marital-status", "relationship", "race", "sex"])

In [3]:
def occupation_mapping(occupation: str) -> str:
    if occupation.strip() in ["Adm-clerical", "Exec-managerial", "Prof-specialty", "Sales", "Tech-support"]:
        return "Office-Worker"
    elif occupation.strip() in ["Armed-Forces", "Protective-serv"]:
        return "Security-Worker"
    elif occupation.strip() in ["Craft-repair", "Machine-op-inspct", "Transport-moving"]:
        return "Machinery-Worker"
    elif occupation.strip() in ["Farming-fishing", "Handlers-cleaners", "Priv-house-serv"]:
        return "Manual-Worker"
    elif occupation.strip() == "Other-service":
        return "Other-Service"

In [4]:
def summarize_workclass(workclass: str) -> str:
    if workclass.strip() in ["Federal-gov", "Local-gov", "State-gov"]:
        return "GOV"
    elif workclass.strip() in ["Private", "Self-emp-inc", "Self-emp-not-inc"]:
        return "PRIVATE"
    else:
        return "UNPAID"

In [5]:
base_adult = pd.read_csv(filepath_or_buffer="adult\\adult.data")
base_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [6]:
base_adult.drop(base_adult.loc[base_adult.workclass == " ?"].index, inplace=True)

In [7]:
base_adult.drop(base_adult.loc[base_adult.occupation == " ?"].index, inplace=True)

In [8]:
base_adult["occupation"] = base_adult["occupation"].apply(func=occupation_mapping)
base_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Office-Worker,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Office-Worker,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Manual-Worker,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Manual-Worker,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Office-Worker,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Office-Worker,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machinery-Worker,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Office-Worker,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Office-Worker,Own-child,White,Male,0,0,20,United-States,<=50K


In [9]:
#base_adult["workclass"] = base_adult["workclass"].apply(func=summarize_workclass)
base_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Office-Worker,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Office-Worker,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Manual-Worker,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Manual-Worker,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Office-Worker,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Office-Worker,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machinery-Worker,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Office-Worker,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Office-Worker,Own-child,White,Male,0,0,20,United-States,<=50K


In [10]:
base_adult = drop_unnecessary(df=base_adult)
base_adult

,age,workclass,fnlwgt,education-num,occupation,capital-gain,capital-loss,hours-per-week,income
0,39,State-gov,77516,13,Office-Worker,2174,0,40,<=50K
1,50,Self-emp-not-inc,83311,13,Office-Worker,0,0,13,<=50K
2,38,Private,215646,9,Manual-Worker,0,0,40,<=50K
3,53,Private,234721,7,Manual-Worker,0,0,40,<=50K
4,28,Private,338409,13,Office-Worker,0,0,40,<=50K
...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,12,Office-Worker,0,0,38,<=50K
32557,40,Private,154374,9,Machinery-Worker,0,0,40,>50K
32558,58,Private,151910,9,Office-Worker,0,0,40,<=50K
32559,22,Private,201490,9,Office-Worker,0,0,20,<=50K


In [11]:
X = base_adult.drop(columns=["income"])
y = base_adult["income"]

In [12]:
#income_replacer = {" <=50K": 0, " >50K": 1}
#y.replace(income_replacer, inplace=True)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
X_train

,age,workclass,fnlwgt,education-num,occupation,capital-gain,capital-loss,hours-per-week
21108,30,Private,158200,15,Office-Worker,0,0,40
17126,49,Private,423222,14,Office-Worker,0,0,50
17019,24,Private,187717,13,Office-Worker,0,0,40
18198,31,Self-emp-not-inc,117346,10,Office-Worker,0,0,48
7431,46,Private,295791,9,Office-Worker,0,0,30
...,...,...,...,...,...,...,...,...
5235,22,Private,271274,7,Office-Worker,0,0,40
26362,32,Private,79870,10,Office-Worker,2597,0,40
427,23,Private,204653,9,Manual-Worker,0,0,72
8862,46,Private,117849,14,Office-Worker,0,0,55


In [14]:
numeric_preprocessor = Pipeline(
    steps=[
        ("scaler", MinMaxScaler())
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("onehot", OneHotEncoder())
    ]
)

In [15]:
cat_variables = ["workclass", "occupation"]
num_variables = ["age", "fnlwgt", "education-num", "hours-per-week", "capital-loss", "capital-gain"]

In [16]:
knn_preprocessor = ColumnTransformer([
    ("categorical", categorical_preprocessor, cat_variables),
    ("numerical", numeric_preprocessor, num_variables)
])
knn_pipeline = make_pipeline(knn_preprocessor, KNeighborsClassifier())
knn_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('kneighborsclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the outpu

In [17]:
knn_column_tranform = make_column_transformer(
    (MinMaxScaler(), num_variables),
    (OneHotEncoder(), cat_variables)
)

knn = KNeighborsClassifier()
knn_pipe = Pipeline(steps=[("colum_transforms", knn_column_tranform), ("knn", knn)])

In [18]:
param_grid = {
    'knn__n_neighbors': [2,5,15, 30, 45, 64],
    "knn__weights": ["uniform", "distance"]
}

grid = GridSearchCV(knn_pipe, param_grid, cv=10, scoring='accuracy')

In [19]:
grid.fit(X_train, y_train)

grid.best_params_

{'knn__n_neighbors': 15, 'knn__weights': 'uniform'}

In [25]:
scores_knn = cross_val_score(grid, X_test, y_test, cv=5)
scores_knn

array([0.79934924, 0.79815518, 0.80412371, 0.79164406, 0.78730331])